# Tool calling against FHIR data: worked solution

Worked solution for the exercise in the lesson **[Tool calling against FHIR data](https://ramkumarmd.com/medical-ai/agentic-ai/fhir-tool-calling)** (*Agentic AI* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](02-fhir-tool-calling.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** FHIR gives agents a standard way to read health records: resources such as Observation and MedicationRequest, searched through web APIs. Safe agent tools are narrow (one question each), read-only by default, scoped to the patient the clinician has open rather than chosen by the model, return values with their source resource and date, and log every call. Authorisation frameworks such as SMART on FHIR limit what an app can see in the first place.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Agents and tool use](https://ramkumarmd.com/medical-ai/agentic-ai/agents-and-tool-use)**: Two tools: an eGFR calculator and a protocol lookup.

In [ ]:
import json

# Tools the agent may call, described in the JSON-schema style most model APIs use
TOOLS = {
    "egfr_ckd_epi_2021": {
        "description": "Estimate eGFR (mL/min/1.73m2) from serum creatinine (mg/dL), age and sex, "
                       "CKD-EPI 2021 equation.",
        "parameters": {"creatinine_mg_dl": "number", "age": "integer", "sex": "'female' or 'male'"},
    },
    "lookup_protocol": {
        "description": "Return the clinic protocol text for a topic.",
        "parameters": {"topic": "one of 'metformin', 'hypoglycaemia'"},
    },
}

def egfr_ckd_epi_2021(creatinine_mg_dl, age, sex):
    k, a = (0.7, -0.241) if sex == "female" else (0.9, -0.302)
    ratio = creatinine_mg_dl / k
    egfr = (142 * min(ratio, 1) ** a * max(ratio, 1) ** -1.200 * 0.9938 ** age
            * (1.012 if sex == "female" else 1))
    return round(egfr)

def lookup_protocol(topic):
    return {"metformin": "Do not start metformin if eGFR is below 45. Stop it if eGFR falls below 30.",
            "hypoglycaemia": "Treat glucose below 70 mg/dL with 15 g fast-acting carbohydrate."}[topic]

print(json.dumps(TOOLS["egfr_ckd_epi_2021"], indent=1))
print(egfr_ckd_epi_2021(1.4, 68, "female"))

# {
#  "description": "Estimate eGFR (mL/min/1.73m2) from serum creatinine (mg/dL), age and sex, CKD-EPI 2021 equation.",
#  "parameters": {
#   "creatinine_mg_dl": "number",
#   "age": "integer",
#   "sex": "'female' or 'male'"
#  }
# }
# 41

From **[Agents and tool use](https://ramkumarmd.com/medical-ai/agentic-ai/agents-and-tool-use)**: Continuing from the previous block: a minimal agent loop with an allow-list and a step limit.

In [ ]:
def scripted_model(messages):
    """Stand-in for a language model. A real model decides these steps itself;
    here they are fixed so the loop runs without an API."""
    tool_results = [m for m in messages if m["role"] == "tool"]
    if len(tool_results) == 0:
        return {"tool_call": {"name": "egfr_ckd_epi_2021",
                              "arguments": {"creatinine_mg_dl": 1.4, "age": 68, "sex": "female"}}}
    if len(tool_results) == 1:
        return {"tool_call": {"name": "lookup_protocol", "arguments": {"topic": "metformin"}}}
    egfr = tool_results[0]["content"]
    return {"answer": f"Estimated eGFR is {egfr} mL/min/1.73m2. The protocol says not to start metformin "
                      f"below 45, so it should not be started; please confirm with the prescriber. "
                      f"[tools: egfr_ckd_epi_2021, lookup_protocol]"}

IMPLEMENTATIONS = {"egfr_ckd_epi_2021": egfr_ckd_epi_2021, "lookup_protocol": lookup_protocol}

def run_agent(question, max_steps=5):
    messages = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        reply = scripted_model(messages)
        if "answer" in reply:
            return reply["answer"], messages
        call = reply["tool_call"]
        if call["name"] not in IMPLEMENTATIONS:                      # allow-list
            raise ValueError(f"Unknown tool {call['name']}")
        result = IMPLEMENTATIONS[call["name"]](**call["arguments"])
        print(f"step {step}: {call['name']}({call['arguments']}) -> {result}")
        messages.append({"role": "tool", "name": call["name"], "content": result})
    raise RuntimeError("Step limit reached without an answer")

answer, log = run_agent("Synthetic case: 68-year-old woman, creatinine 1.4 mg/dL. Can metformin be started?")
print("\n" + answer)

# step 1: egfr_ckd_epi_2021({'creatinine_mg_dl': 1.4, 'age': 68, 'sex': 'female'}) -> 41
# step 2: lookup_protocol({'topic': 'metformin'}) -> Do not start metformin if eGFR is below 45. Stop it if eGFR falls below 30.
#
# Estimated eGFR is 41 mL/min/1.73m2. The protocol says not to start metformin below 45, so it should not be started; please confirm with the prescriber. [tools: egfr_ckd_epi_2021, lookup_protocol]

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A synthetic FHIR bundle

Two synthetic patients, three observations and two medication requests.

In [ ]:
# A small synthetic FHIR R4 bundle (made-up patients, no real data)
bundle = {"resourceType": "Bundle", "entry": [{"resource": r} for r in [
    {"resourceType": "Patient", "id": "p1", "birthDate": "1958-04-02", "gender": "female"},
    {"resourceType": "Patient", "id": "p2", "birthDate": "1971-09-15", "gender": "male"},
    {"resourceType": "Observation", "id": "o1", "subject": {"reference": "Patient/p1"},
     "code": {"coding": [{"system": "http://loinc.org", "code": "4548-4", "display": "Hemoglobin A1c"}]},
     "effectiveDateTime": "2026-01-10", "valueQuantity": {"value": 8.2, "unit": "%"}},
    {"resourceType": "Observation", "id": "o2", "subject": {"reference": "Patient/p1"},
     "code": {"coding": [{"system": "http://loinc.org", "code": "4548-4", "display": "Hemoglobin A1c"}]},
     "effectiveDateTime": "2026-07-02", "valueQuantity": {"value": 7.4, "unit": "%"}},
    {"resourceType": "Observation", "id": "o3", "subject": {"reference": "Patient/p1"},
     "code": {"coding": [{"system": "http://loinc.org", "code": "98979-8", "display": "eGFR (CKD-EPI 2021)"}]},
     "effectiveDateTime": "2026-07-02", "valueQuantity": {"value": 41, "unit": "mL/min/1.73m2"}},
    {"resourceType": "MedicationRequest", "id": "m1", "status": "active", "subject": {"reference": "Patient/p1"},
     "medicationCodeableConcept": {"text": "Metformin 500 mg tablets"}, "authoredOn": "2024-05-01"},
    {"resourceType": "MedicationRequest", "id": "m2", "status": "stopped", "subject": {"reference": "Patient/p1"},
     "medicationCodeableConcept": {"text": "Gliclazide 80 mg tablets"}, "authoredOn": "2023-02-11"},
]]}
resources = [e["resource"] for e in bundle["entry"]]
print({t: sum(r["resourceType"] == t for r in resources)
       for t in ["Patient", "Observation", "MedicationRequest"]})

# {'Patient': 2, 'Observation': 3, 'MedicationRequest': 2}

### Narrow, read-only, patient-scoped tools

Continuing from the bundle block: patient-scoped tools with provenance and an audit log.

In [ ]:
class PatientTools:
    """Read-only tools scoped to ONE patient, chosen by the clinician, not by the model."""
    def __init__(self, resources, patient_id):
        self.patient_ref = f"Patient/{patient_id}"
        self.resources = resources
        self.audit = []

    def _mine(self, resource_type):
        return [r for r in self.resources if r["resourceType"] == resource_type
                and r.get("subject", {}).get("reference") == self.patient_ref]

    def latest_observation(self, loinc_code):
        self.audit.append(("latest_observation", loinc_code))
        matches = [o for o in self._mine("Observation")
                   if any(c["code"] == loinc_code for c in o["code"]["coding"])]
        if not matches:
            return {"result": None, "note": f"no observation with LOINC {loinc_code}"}
        o = max(matches, key=lambda o: o["effectiveDateTime"])
        return {"value": o["valueQuantity"]["value"], "unit": o["valueQuantity"]["unit"],
                "date": o["effectiveDateTime"], "source": f"Observation/{o['id']}"}

    def active_medications(self):
        self.audit.append(("active_medications", None))
        return [{"medication": m["medicationCodeableConcept"]["text"],
                 "source": f"MedicationRequest/{m['id']}"}
                for m in self._mine("MedicationRequest") if m["status"] == "active"]

tools = PatientTools(resources, patient_id="p1")
print(tools.latest_observation("4548-4"))
print(tools.latest_observation("98979-8"))
print(tools.active_medications())
print(tools.latest_observation("2093-3"))
print("audit log:", tools.audit)

# {'value': 7.4, 'unit': '%', 'date': '2026-07-02', 'source': 'Observation/o2'}
# {'value': 41, 'unit': 'mL/min/1.73m2', 'date': '2026-07-02', 'source': 'Observation/o3'}
# [{'medication': 'Metformin 500 mg tablets', 'source': 'MedicationRequest/m1'}]
# {'result': None, 'note': 'no observation with LOINC 2093-3'}
# audit log: [('latest_observation', '4548-4'), ('latest_observation', '98979-8'), ('active_medications', None), ('latest_observation', '2093-3')]

## Exercise

**Goal:** Build and test patient-scoped FHIR tools over synthetic data, then connect them to the agent loop.

**Data:** Synthetic FHIR bundle in the notebook; optional Synthea FHIR R4 bundles (Generated in the notebook; Synthea is Apache 2.0)

### Step 1

Reproduce the tools and add one for active conditions (Condition resources) and one for allergies.

In [ ]:
import datetime

extra = [
    {"resourceType": "Condition", "id": "c1", "subject": {"reference": "Patient/p1"},
     "clinicalStatus": {"coding": [{"code": "active"}]},
     "code": {"coding": [{"system": "http://snomed.info/sct", "code": "44054006", "display": "Type 2 diabetes mellitus"}]}},
    {"resourceType": "Condition", "id": "c2", "subject": {"reference": "Patient/p1"},
     "clinicalStatus": {"coding": [{"code": "resolved"}]},
     "code": {"coding": [{"system": "http://snomed.info/sct", "code": "195662009", "display": "Acute viral pharyngitis"}]}},
    # AllergyIntolerance points to the patient with "patient", not "subject"
    {"resourceType": "AllergyIntolerance", "id": "a1", "patient": {"reference": "Patient/p1"},
     "code": {"text": "Penicillin"}, "reaction": [{"description": "Rash"}]},
    {"resourceType": "Observation", "id": "o4", "subject": {"reference": "Patient/p1"},
     "code": {"coding": [{"system": "http://loinc.org", "code": "13457-7", "display": "LDL cholesterol (calculated)"}]},
     "effectiveDateTime": "2025-03-14", "valueQuantity": {"value": 3.1, "unit": "mmol/L"}},
    {"resourceType": "Observation", "id": "o5", "subject": {"reference": "Patient/p2"},
     "code": {"coding": [{"system": "http://loinc.org", "code": "4548-4", "display": "Hemoglobin A1c"}]},
     "effectiveDateTime": "2026-08-01", "valueQuantity": {"value": 9.6, "unit": "%"}},
]
all_resources = resources + extra

class MoreTools(PatientTools):
    def _mine(self, resource_type):
        return [r for r in self.resources if r["resourceType"] == resource_type
                and (r.get("subject") or r.get("patient") or {}).get("reference") == self.patient_ref]

    def active_conditions(self):
        self.audit.append(("active_conditions", None))
        return [{"condition": c["code"]["coding"][0]["display"], "snomed": c["code"]["coding"][0]["code"],
                 "source": f"Condition/{c['id']}"}
                for c in self._mine("Condition") if c["clinicalStatus"]["coding"][0]["code"] == "active"]

    def allergies(self):
        self.audit.append(("allergies", None))
        return [{"substance": a["code"]["text"], "reaction": a.get("reaction", [{}])[0].get("description"),
                 "source": f"AllergyIntolerance/{a['id']}"} for a in self._mine("AllergyIntolerance")]

t1 = MoreTools(all_resources, "p1")
print(t1.active_conditions())
print(t1.allergies())

The lesson's `_mine` only looked at `subject`, so it would have returned *no allergies* for every patient,
with no error. A tool that silently returns an empty list is more dangerous than one that fails; test each tool
against a record where you know the answer.

### Step 2

Add a 'stale' flag when the latest observation is more than 12 months old.

In [ ]:
TODAY = datetime.date(2026, 10, 18)      # fixed so the result is reproducible

class StaleAwareTools(MoreTools):
    def latest_observation(self, loinc_code, max_age_days=365):
        result = super().latest_observation(loinc_code)
        if result.get("date"):
            age = (TODAY - datetime.date.fromisoformat(result["date"][:10])).days   # date part of a FHIR dateTime
            result["days_old"] = age
            result["stale"] = age > max_age_days
        return result

t2 = StaleAwareTools(all_resources, "p1")
for code in ["4548-4", "13457-7"]:
    print(t2.latest_observation(code))

### Step 3

Try to read patient p2's data through the p1 tools and confirm it is impossible.

In [ ]:
t_p1 = StaleAwareTools(all_resources, "p1")
print("p1 HbA1c:", t_p1.latest_observation("4548-4"))
try:
    t_p1.latest_observation("4548-4", patient_id="p2")          # the model tries to choose the patient
except TypeError as e:
    print("Asking for p2 through the p1 tools fails:", e)
print("Observation o5 belongs to p2 and exists in the bundle:", any(r["id"] == "o5" for r in all_resources))
print("Can the p1 tools see it?", any(t_p1.latest_observation(c).get("source") == "Observation/o5" for c in ["4548-4"]))

The patient is fixed when the tools are created, by the clinician's context, and no tool takes a patient
argument, so there is nothing for a model (or an injected instruction) to change. Scoping is a property of the
code, which can be tested, rather than an instruction the model is asked to follow.

### Step 4

Plug the tools into the agent loop from the previous lesson and have the scripted model produce a summary with source references.

In [ ]:
def summary_model(messages):
    """Scripted model that gathers facts with the tools, then writes a summary citing every source."""
    plan = [("latest_observation", {"loinc_code": "4548-4"}), ("latest_observation", {"loinc_code": "98979-8"}),
            ("latest_observation", {"loinc_code": "13457-7"}), ("active_medications", {}),
            ("active_conditions", {}), ("allergies", {})]
    done = [m for m in messages if m["role"] == "tool"]
    if len(done) < len(plan):
        name, args = plan[len(done)]
        return {"tool_call": {"name": name, "arguments": args}}
    r = {i: m["content"] for i, m in enumerate(done)}
    meds = "; ".join(f"{m['medication']} [{m['source']}]" for m in r[3])
    conds = "; ".join(f"{c['condition']} [{c['source']}]" for c in r[4])
    allergies = "; ".join(f"{a['substance']} ({a['reaction']}) [{a['source']}]" for a in r[5])
    ldl = r[2]
    return {"answer": "\n".join([
        "Pre-clinic summary (synthetic patient p1) - draft for clinician review",
        f"- Conditions: {conds}",
        f"- HbA1c {r[0]['value']}{r[0]['unit']} on {r[0]['date']} [{r[0]['source']}]",
        f"- eGFR {r[1]['value']} {r[1]['unit']} on {r[1]['date']} [{r[1]['source']}]",
        f"- LDL {ldl['value']} {ldl['unit']} on {ldl['date']} [{ldl['source']}]" + (" - STALE, over 12 months old" if ldl["stale"] else ""),
        f"- Active medicines: {meds}",
        f"- Allergies: {allergies}",
    ])}

def run_with_tools(question, model, tool_obj, allowed, max_steps=10):
    messages = [{"role": "user", "content": question}]
    for _ in range(max_steps):
        reply = model(messages)
        if "answer" in reply:
            return reply["answer"], messages
        call = reply["tool_call"]
        if call["name"] not in allowed:
            raise ValueError(f"tool {call['name']} not allowed")
        messages.append({"role": "tool", "name": call["name"], "content": getattr(tool_obj, call["name"])(**call["arguments"])})
    raise RuntimeError("step limit reached")

ALLOWED = {"latest_observation", "active_medications", "active_conditions", "allergies"}
summary, _ = run_with_tools("Prepare a pre-clinic summary.", summary_model, StaleAwareTools(all_resources, "p1"), ALLOWED)
print(summary)

Every fact carries the FHIR resource it came from, so a clinician can check any line in one click, and stale
results are labelled rather than presented as current. The checks in the next lesson rely on those references.

### Step 5

Optional: load a Synthea FHIR bundle and run the same tools on it.

In [ ]:
# Optional: the same tools on a Synthea FHIR R4 bundle (synthetic patients, Apache 2.0; about 85 MB download).
import json
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

URL = "https://synthetichealth.github.io/synthea-sample-data/downloads/synthea_sample_data_fhir_r4_sep2019.zip"
zip_path = Path("data/synthea_fhir_r4_sep2019.zip")
zip_path.parent.mkdir(exist_ok=True)
if not zip_path.exists():
    urlretrieve(URL, zip_path)

with zipfile.ZipFile(zip_path) as z:
    name = next(n for n in sorted(z.namelist()) if n.endswith(".json") and "Information" not in n)
    synthea = [e["resource"] for e in json.load(z.open(name))["entry"]]

# Synthea references resources as "urn:uuid:<id>"; rewrite them to the "Patient/<id>" form the tools expect
patient = next(r for r in synthea if r["resourceType"] == "Patient")
for r in synthea:
    for key in ("subject", "patient"):
        if r.get(key, {}).get("reference", "").startswith("urn:uuid:"):
            r[key] = {"reference": "Patient/" + r[key]["reference"].removeprefix("urn:uuid:")}
real_tools = StaleAwareTools(synthea, patient["id"])
print("HbA1c:", real_tools.latest_observation("4548-4"))
print("Glucose (2339-0):", real_tools.latest_observation("2339-0"))
print("Active conditions:", [c["condition"] for c in real_tools.active_conditions()][:6])
from collections import Counter
print("Active medications:", len(real_tools.active_medications()),
      "| MedicationRequest statuses in this bundle:", dict(Counter(r["status"] for r in synthea if r["resourceType"] == "MedicationRequest")))
print("Allergies:", real_tools.allergies())

Real bundles expose the assumptions in toy tools: Synthea writes references as `urn:uuid:` rather than
`Patient/<id>`, dates carry a time and time zone ("2019-04-22T07:22:41-04:00", which first broke the stale
check), and many results are years old (hence the stale flag). Each mismatch needs handling explicitly; a tool should fail loudly on a
resource shape it does not understand.

---

Lesson: [https://ramkumarmd.com/medical-ai/agentic-ai/fhir-tool-calling](https://ramkumarmd.com/medical-ai/agentic-ai/fhir-tool-calling). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.